In [ ]:
# !pip install tensorflow -q

In [ ]:
import os
import pandas as pd
import numpy as np  
import itertools
import numpy.typing as npt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from tensorflow.keras.models import Sequential, Model 
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import datetime
from scipy.signal import find_peaks, peak_prominences


# Change directory as required
os.getcwd()
os.chdir("C:\\path\\to\\CNC\\folder\\")

In [ ]:
def ts_train_test_split(
    ts: npt.ArrayLike, training_size: int
) -> tuple[pd.DataFrame, pd.DataFrame] | tuple[np.ndarray, np.ndarray]:
    """
    Time series train test split. Performs a single split of the series.

    Parameters:
    ----------
    ts: array-like
        univariate time series data set

    training_size: int
        Size of the training set. The test set length

    Returns:
    -------
    Tuple[pd.DataFrame, pd.DataFrame] | Tuple[np.ndarray, np.ndarray]
        A tuple containing the training and test sets,
        either as DataFrames or NumPy arrays
    """
    if training_size >= len(ts):
        raise ValueError("training_size must be < length of series")

    if isinstance(ts, pd.DataFrame):
        return ts.iloc[:training_size], ts.iloc[training_size:]

    return np.asarray(ts[:training_size]), np.asarray(ts[training_size:])

In [ ]:
path = 'datasets_pseudo/Threshold/'
names = os.listdir(path)
date_str = datetime.datetime.now().strftime("%m%d%Y_%H%M%S")
dfs = {}

for name in names:
    machine, material, component, _ = name.replace('.csv', '').split('_')
    tempdf = pd.read_csv(f"{path}\\{name}", index_col=0) 
    print(f"Size of {name}: {tempdf.shape}")

    dfs[name.replace('.csv', '')] = tempdf

df = pd.concat(dfs.values(), ignore_index=True)


In [ ]:
columns_to_predict = ['CURRENT|1', 'CURRENT|2', 'CURRENT|3', 'CURRENT|6'][:1]

latent_dims = [8, 10, 15, 20]
contamination_values = [0.01, 0.05, 0.02]
epochs_list = [50, 75, 100, 150, 200, 250]
batch_sizes = [32, 64, 128]

feature_sets = {
    'all_sensors': list(set(df.columns) - set(['Machine', 'Material', 'Component']) - set(['CURRENT|1_Peak', 'CURRENT|2_Peak', 'CURRENT|3_Peak', 'CURRENT|6_Peak'])), 
    'torques' : ['TORQUE|1', 'TORQUE|2', 'TORQUE|3', 'TORQUE|6'], 
    'cmd_speed' : ['CMD_SPEED|1', 'CMD_SPEED|2', 'CMD_SPEED|3', 'CMD_SPEED|6']
}

# adjust as required. Was set as a constant since the computation takes a long time to run 
numeric_features = feature_sets['all_sensors']

plot_path = 'Plots/For Presentation/'

In [ ]:
# add datasets names to be considered. was limited to create visualisations for the presentation
visualize = ['CMX1_S_CP1', 'DMC2_AL_CP2']

results = []

for name, target_column in list(itertools.product(names, columns_to_predict)):

    axis = target_column[-2:]
    numeric_features_ = [x for x in numeric_features if axis in x]

    if name.replace("threshold_", "").replace('.csv', '') in visualize:
            
        print(f'Training for column {target_column}')
        pseudo_label_col = f"{target_column}_Peak"

        data = pd.read_csv(f"{path}\\{name}")  #, index_col=0
        full_index = data.index
        target_full = data[[target_column]]

        train_size = round(data.shape[0]*0.7)
        X_train, X_test = ts_train_test_split(data[numeric_features_], train_size)

        y_train, y_test = ts_train_test_split(data[[pseudo_label_col]], train_size)
        y_train_pseudo = y_train.astype(bool).values.flatten()
        y_test_pseudo = y_test.astype(bool).values.flatten()

        scaler_X = MinMaxScaler()
        X_train = scaler_X.fit_transform(X_train)
        X_test = scaler_X.transform(X_test)
        X_full = scaler_X.transform(data[numeric_features_]) 

        
        for latent_dim, contamination, epochs, batch_size in list(itertools.product(latent_dims, contamination_values, epochs_list, batch_sizes)):
            
            print(f'training for dims = {latent_dim}, contamination = {contamination}, epochs = {epochs}, batch = {batch_size}')

            os.makedirs(f'{plot_path}\\AE\\{latent_dim}_{contamination}_{epochs}_{batch_size}\\', exist_ok=True)

            input_dim = X_train.shape[1]
            autoencoder = Sequential([
                Dense(64, activation='elu', input_shape=(input_dim,)),
                Dense(32, activation='elu'),
                Dense(latent_dim, activation='linear', name='bottleneck'),
                Dense(32, activation='elu'),
                Dense(64, activation='elu'),
                Dense(input_dim, activation='sigmoid')
            ])
            autoencoder.compile(optimizer=Adam(), loss='mse')

            autoencoder.fit(X_train, X_train, epochs=epochs, batch_size=batch_size,
                            validation_data=(X_test, X_test), verbose=0)

            X_recon = autoencoder.predict(X_full)
            recon_errors = np.mean((X_full - X_recon) ** 2, axis=1)

            threshold = np.quantile(recon_errors, 1 - contamination)
            peaks_detected = recon_errors > threshold

            # Evaluate (test portion to compare with pseudo labels)
            recon_errors_test = recon_errors[train_size:]
            peaks_test = peaks_detected[train_size:]
            precision = precision_score(y_test_pseudo, peaks_test, zero_division=0)
            recall = recall_score(y_test_pseudo, peaks_test, zero_division=0)
            f1 = f1_score(y_test_pseudo, peaks_test, zero_division=0)

            n_detected = np.sum(peaks_detected)
            if n_detected > 0:
                anomaly_values = target_full.iloc[peaks_detected].values.flatten()
                avg_height = np.mean(np.abs(anomaly_values))

                rel_indices = np.where(peaks_detected)[0]
                prominences, _, _ = peak_prominences(target_full.values.flatten(), rel_indices)
                avg_prominence = prominences.mean() if len(prominences) > 0 else 0.0
            else:
                avg_height, avg_prominence = 0.0, 0.0

            results.append({
                'Dataset': name.replace(".csv", ""), 
                'Column': target_column,
                'Latent_Dim': latent_dim,
                'Contamination': contamination,
                'Epochs': epochs,
                'Batch size': batch_size,
                'Precision': precision,
                'Recall': recall,
                'F1': f1,
                'Detected_Peaks': n_detected,
                'Avg_Abs_Height': avg_height,
                'Avg_Prominence': avg_prominence
            })

            print(f"Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}, Peaks={n_detected}")

            plt.figure(figsize=(10, 4))
            plt.plot(full_index, target_full, label='Current-Signal')
            plt.scatter(
                full_index[peaks_detected],
                target_full.iloc[peaks_detected],
                color='red', marker='x', label='Detected Peaks'
            )
            plt.legend(fontsize=12)
            plt.grid(True)
            plt.tick_params(axis='both', labelsize=12)
            plt.savefig(f'{plot_path}\\AE\\{latent_dim}_{contamination}_{epochs}_{batch_size}\\AE_{target_column.replace("|", "")}_{name.replace("threshold_", "")}.png', dpi=300, bbox_inches='tight')
            plt.show()


results_df = pd.DataFrame(results)
results_df.sort_values(by='F1', ascending=False, inplace=True)
results_df.to_csv(f"Results/IT_results_{date_str}_all.csv")

